# 🔍 Finding a Kanté replacement across the Big 5

**Competition:** Big 5 leagues 2015/16   
**Dataset:** StatsBomb Open Data (events and lineups via `statsbombpy`)  
**Purpose:** Build a midfielder pool from open events, find Kanté's closest twins, print the numbers used on the deck, and export title-free charts into `slides/kante-replacement/assets/`.  
**Methods:** Season match loop, SkillCorner-inspired midfielder metrics from open events (no tracking, xPass, EPV, or line-breaking options), Twin Test style/quality similarity.  
**Author:** [Oscar Yu](https://www.linkedin.com/in/oscar-yu-cheuk-chun/)  

---



# 1. Imports and setup

Resolve paths from the repo root, pin the 2015/16 Big 5 season ids, and keep only the event columns needed for the midfielder features.


In [2]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from statsbombpy import sb

try:
    from statsbombpy.api_client import NoAuthWarning

    warnings.filterwarnings("ignore", category=NoAuthWarning)
except ImportError:
    pass

# Repo root = football-slides/. Works when cwd is the repo or projects/.
NOTEBOOK_DIR = Path.cwd().resolve()
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (candidate / "slides").is_dir() and (candidate / "projects").is_dir():
        ROOT = candidate
        break
else:
    ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "projects" else NOTEBOOK_DIR

DATA_DIR = ROOT / "data" / "kante-replacement"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MATCHES_PATH = DATA_DIR / "matches.parquet"
LINEUPS_PATH = DATA_DIR / "lineups.parquet"
EVENTS_PATH = DATA_DIR / "events.parquet"
MANIFEST_PATH = DATA_DIR / "build_manifest.json"

# Only open-data season with all five Big 5 leagues.
SEASONS = [
    {"competition_id": 2, "season_id": 27, "competition_name": "Premier League", "season_name": "2015/2016"},
    {"competition_id": 11, "season_id": 27, "competition_name": "La Liga", "season_name": "2015/2016"},
    {"competition_id": 9, "season_id": 27, "competition_name": "1. Bundesliga", "season_name": "2015/2016"},
    {"competition_id": 12, "season_id": 27, "competition_name": "Serie A", "season_name": "2015/2016"},
    {"competition_id": 7, "season_id": 27, "competition_name": "Ligue 1", "season_name": "2015/2016"},
]

# Columns kept for later midfielder feature work. Everything else is dropped on load.
EVENT_COLS = [
    "match_id",
    "id",
    "index",
    "period",
    "timestamp",
    "minute",
    "second",
    "type",
    "possession",
    "possession_team_id",
    "possession_team",
    "play_pattern",
    "team_id",
    "team",
    "player_id",
    "player",
    "position",
    "location",
    "under_pressure",
    "counterpress",
    "duration",
    "pass_end_location",
    "pass_outcome",
    "pass_length",
    "pass_angle",
    "pass_height",
    "pass_body_part",
    "pass_type",
    "pass_cross",
    "pass_cut_back",
    "pass_switch",
    "pass_through_ball",
    "pass_goal_assist",
    "pass_shot_assist",
    "carry_end_location",
    "dribble_outcome",
    "duel_type",
    "duel_outcome",
    "interception_outcome",
    "ball_receipt_outcome",
    "shot_outcome",
    "shot_statsbomb_xg",
    "foul_committed_card",
    "bad_behaviour_card",
]

MATCH_COLS = [
    "match_id",
    "match_date",
    "kick_off",
    "competition",
    "season",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "competition_id",
    "season_id",
    "competition_name",
    "season_name",
]

pd.set_option("display.max_columns", None)
print(f"ROOT = {ROOT}")
print(f"DATA_DIR = {DATA_DIR}")



ROOT = C:\Users\oscar\Documents\GitHub\Oscar-Yu-Football-Analysis
DATA_DIR = C:\Users\oscar\Documents\GitHub\Oscar-Yu-Football-Analysis\projects\data\kante-replacement


# 2. Helpers

Boolean cleanup and lineup / event flatteners for the download loop.


In [3]:
def as_boolean_series(frame: pd.DataFrame, col: str) -> pd.Series:
    if col not in frame.columns:
        return pd.Series(False, index=frame.index)
    series = frame[col]
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype("string").str.lower().isin(["true", "1", "yes"])


def _label_from_obj(value):
    if isinstance(value, dict):
        return value.get("competition_name") or value.get("season_name") or value.get("name")
    return value


def flatten_lineups(lineups: dict, match_id: int, competition_id: int, season_id: int) -> pd.DataFrame:
    parts = []
    for team_name, team in lineups.items():
        part = team.copy()
        part["team_name"] = team_name
        part["match_id"] = match_id
        part["competition_id"] = competition_id
        part["season_id"] = season_id
        parts.append(part)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


def trim_events(events: pd.DataFrame, match_id: int, competition_id: int, season_id: int) -> pd.DataFrame:
    frame = events.copy()
    frame["match_id"] = match_id
    frame["competition_id"] = competition_id
    frame["season_id"] = season_id
    keep = [col for col in EVENT_COLS + ["competition_id", "season_id"] if col in frame.columns]
    frame = frame.loc[:, keep].copy()
    for flag in ("under_pressure", "counterpress"):
        if flag in frame.columns:
            frame[flag] = as_boolean_series(frame, flag)
    return frame


def load_season_matches(competition_id: int, season_id: int, competition_name: str, season_name: str) -> pd.DataFrame:
    matches = sb.matches(competition_id=competition_id, season_id=season_id).copy()
    matches["competition_id"] = competition_id
    matches["season_id"] = season_id
    matches["competition_name"] = competition_name
    matches["season_name"] = season_name
    if "competition" in matches.columns:
        matches["competition"] = matches["competition"].map(_label_from_obj)
    if "season" in matches.columns:
        matches["season"] = matches["season"].map(_label_from_obj)
    keep = [col for col in MATCH_COLS if col in matches.columns]
    return matches.loc[:, keep].copy()


def load_match(match_id: int, competition_id: int, season_id: int):
    lineups = flatten_lineups(sb.lineups(match_id=match_id), match_id, competition_id, season_id)
    events = trim_events(sb.events(match_id=match_id), match_id, competition_id, season_id)
    return lineups, events


def serialize_nested(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Store list/dict cells as JSON strings so parquet write stays stable."""
    out = frame.copy()
    for col in columns:
        if col not in out.columns:
            continue
        out[col] = out[col].map(
            lambda value: json.dumps(value) if isinstance(value, (list, dict)) else value
        )
    return out


# 3. Build loop

Download all Big 5 2015/16 matches, lineups, and events. Skip when parquet already exists.


In [3]:
match_frames = []
lineup_frames = []
event_frames = []
failed = []
loaded_matches = 0

for season in SEASONS:
    competition_id = season["competition_id"]
    season_id = season["season_id"]
    label = f"{season['competition_name']} {season['season_name']}"

    try:
        season_matches = load_season_matches(
            competition_id=competition_id,
            season_id=season_id,
            competition_name=season["competition_name"],
            season_name=season["season_name"],
        )
    except Exception as exc:
        failed.append(f"{label} match list: {exc}")
        continue

    match_frames.append(season_matches)
    match_ids = [int(mid) for mid in season_matches["match_id"]]

    for match_id in tqdm(match_ids, desc=label):
        try:
            lineups, events = load_match(match_id, competition_id, season_id)
            loaded_matches += 1
        except Exception as exc:
            failed.append(f"match {match_id} ({label}): {exc}")
            continue

        if not lineups.empty:
            lineup_frames.append(lineups)
        if not events.empty:
            event_frames.append(events)

        del lineups, events

matches = pd.concat(match_frames, ignore_index=True) if match_frames else pd.DataFrame()
lineups = pd.concat(lineup_frames, ignore_index=True) if lineup_frames else pd.DataFrame()
events = pd.concat(event_frames, ignore_index=True) if event_frames else pd.DataFrame()

print(f"Matches listed: {len(matches)}")
print(f"Matches loaded: {loaded_matches}")
print(f"Lineup rows: {len(lineups):,}")
print(f"Event rows: {len(events):,}")
print(f"Failed: {len(failed)}")
if failed[:10]:
    for item in failed[:10]:
        print(f"  {item}")


Premier League 2015/2016:   0%|          | 0/380 [00:00<?, ?it/s]

La Liga 2015/2016:   0%|          | 0/380 [00:00<?, ?it/s]

1. Bundesliga 2015/2016:   0%|          | 0/34 [00:00<?, ?it/s]

Serie A 2015/2016:   0%|          | 0/380 [00:00<?, ?it/s]

Ligue 1 2015/2016:   0%|          | 0/377 [00:00<?, ?it/s]

Matches listed: 1551
Matches loaded: 1551
Lineup rows: 58,883
Event rows: 5,436,699
Failed: 0


# 4. Write database

Persist `matches`, `lineups`, `events`, and a small build manifest.


In [4]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

lineups_out = serialize_nested(lineups, ["positions"])
events_out = serialize_nested(
    events,
    ["location", "pass_end_location", "carry_end_location"],
)

matches.to_parquet(MATCHES_PATH, index=False)
lineups_out.to_parquet(LINEUPS_PATH, index=False)
events_out.to_parquet(EVENTS_PATH, index=False)

manifest = {
    "seasons": SEASONS,
    "matches_listed": int(len(matches)),
    "matches_loaded": int(loaded_matches),
    "lineup_rows": int(len(lineups)),
    "event_rows": int(len(events)),
    "failed_count": int(len(failed)),
    "failed_sample": failed[:50],
    "paths": {
        "matches": str(MATCHES_PATH.relative_to(ROOT)),
        "lineups": str(LINEUPS_PATH.relative_to(ROOT)),
        "events": str(EVENTS_PATH.relative_to(ROOT)),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Wrote {MATCHES_PATH.relative_to(ROOT)}")
print(f"Wrote {LINEUPS_PATH.relative_to(ROOT)}")
print(f"Wrote {EVENTS_PATH.relative_to(ROOT)}")
print(f"Wrote {MANIFEST_PATH.relative_to(ROOT)}")

display(matches.groupby(["competition_name", "season_name"], as_index=False).size())
display(matches.head())
display(lineups.head())
display(events.head())


Wrote projects\data\kante-replacement\matches.parquet
Wrote projects\data\kante-replacement\lineups.parquet
Wrote projects\data\kante-replacement\events.parquet
Wrote projects\data\kante-replacement\build_manifest.json


,competition_name,season_name,size
0,1. Bundesliga,2015/2016,34
1,La Liga,2015/2016,380
2,Ligue 1,2015/2016,377
3,Premier League,2015/2016,380
4,Serie A,2015/2016,380


,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,competition_id,season_id,competition_name,season_name
0,3754217,2015-09-19,11:45:00.000,England - Premier League,2015/2016,Chelsea,Arsenal,2,0,2,27,Premier League,2015/2016
1,3754117,2015-12-13,12:30:00.000,England - Premier League,2015/2016,Aston Villa,Arsenal,0,2,2,27,Premier League,2015/2016
2,3754296,2015-12-21,19:00:00.000,England - Premier League,2015/2016,Arsenal,Manchester City,2,1,2,27,Premier League,2015/2016
3,3753983,2015-10-31,14:00:00.000,England - Premier League,2015/2016,Swansea City,Arsenal,0,3,2,27,Premier League,2015/2016
4,3754160,2015-12-05,14:00:00.000,England - Premier League,2015/2016,Arsenal,Sunderland,3,1,2,27,Premier League,2015/2016


,player_id,player_name,player_nickname,jersey_number,country,cards,positions,team_name,match_id,competition_id,season_id
0,3339,Asmir Begović,None,1,Bosnia and Herzegovina,[],"[{'position_id': 1, 'position': 'Goalkeeper', ...",Chelsea,3754217,2,27
1,3381,Nemanja Matić,None,21,Serbia,[],"[{'position_id': 11, 'position': 'Left Defensi...",Chelsea,3754217,2,27
2,3445,Radamel Falcao García Zárate,Falcao,9,Colombia,[],[],Chelsea,3754217,2,27
3,3456,Kurt Happy Zouma,Kurt Zouma,5,France,[],"[{'position_id': 3, 'position': 'Right Center ...",Chelsea,3754217,2,27
4,3478,Francesc Fàbregas i Soler,Cesc Fàbregas,4,Spain,[],"[{'position_id': 9, 'position': 'Right Defensi...",Chelsea,3754217,2,27


,match_id,id,index,period,timestamp,minute,second,type,possession,possession_team_id,possession_team,play_pattern,team_id,team,player_id,player,position,location,under_pressure,counterpress,duration,pass_end_location,pass_outcome,pass_length,pass_angle,pass_height,pass_body_part,pass_type,pass_cross,pass_cut_back,pass_switch,pass_through_ball,pass_goal_assist,pass_shot_assist,carry_end_location,dribble_outcome,duel_type,duel_outcome,interception_outcome,ball_receipt_outcome,shot_outcome,shot_statsbomb_xg,foul_committed_card,bad_behaviour_card,competition_id,season_id
0,3754217,9d86a178-3514-45d1-9d14-1372e846d17b,1,1,00:00:00.000,0,0,Starting XI,1,33,Chelsea,Regular Play,33,Chelsea,NaN,NaN,NaN,NaN,False,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,27
1,3754217,82acc213-90f3-4fee-8305-bd9e403cec42,2,1,00:00:00.000,0,0,Starting XI,1,33,Chelsea,Regular Play,1,Arsenal,NaN,NaN,NaN,NaN,False,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,27
2,3754217,d18f1a33-65ba-42e5-a9fd-bbfc709694e8,3,1,00:00:00.000,0,0,Half Start,1,33,Chelsea,Regular Play,33,Chelsea,NaN,NaN,NaN,NaN,False,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,27
3,3754217,49147091-455c-4372-8d56-96e1ca71d01a,4,1,00:00:00.000,0,0,Half Start,1,33,Chelsea,Regular Play,1,Arsenal,NaN,NaN,NaN,NaN,False,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,27
4,3754217,7457b6d0-10f8-48ea-9b80-8c0703914ece,1718,2,00:00:00.000,45,0,Half Start,92,1,Arsenal,From Throw In,1,Arsenal,NaN,NaN,NaN,NaN,False,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,27


# 5. Load cached database

Re-read parquet. Start here when iterating on metrics without re-downloading.


In [4]:
import ast
import json
from collections import Counter, defaultdict

import numpy as np

matches = pd.read_parquet(MATCHES_PATH)
lineups = pd.read_parquet(LINEUPS_PATH)

# Full events.parquet is ~6.7 GiB in memory; keep only metric columns (~2.2 GiB).
METRIC_EVENT_COLS = [
    "match_id",
    "index",
    "period",
    "minute",
    "second",
    "type",
    "possession",
    "possession_team_id",
    "team_id",
    "player_id",
    "location",
    "under_pressure",
    "counterpress",
    "duration",
    "pass_end_location",
    "pass_outcome",
    "carry_end_location",
    "dribble_outcome",
    "duel_type",
    "duel_outcome",
    "interception_outcome",
    "competition_id",
    "season_id",
]
events = pd.read_parquet(EVENTS_PATH, columns=METRIC_EVENT_COLS)

PLAYER_METRICS_PATH = DATA_DIR / "player_metrics.parquet"
MIDFIELDERS_PATH = DATA_DIR / "midfielders.parquet"

MINUTES_FLOOR = 900

MIDFIELD_LABELS = {
    "Center Defensive Midfield",
    "Left Defensive Midfield",
    "Right Defensive Midfield",
    "Center Midfield",
    "Left Center Midfield",
    "Right Center Midfield",
    "Center Attacking Midfield",
    "Left Attacking Midfield",
    "Right Attacking Midfield",
    "Left Midfield",
    "Right Midfield",
}

print(matches.groupby(["competition_name", "season_name"]).size())
print(f"lineups={len(lineups):,} events={len(events):,}")


competition_name  season_name
1. Bundesliga     2015/2016       34
La Liga           2015/2016      380
Ligue 1           2015/2016      377
Premier League    2015/2016      380
Serie A           2015/2016      380
dtype: int64
lineups=58,883 events=5,436,699


# 6. Metric helpers

Minutes from lineup position windows and shared event prep.


In [5]:
def _clock_to_seconds(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value)
    if text.lower() in {"none", "nan", ""}:
        return None
    mins, secs = text.split(":")
    return float(mins) * 60 + float(secs)


def parse_xy(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan, np.nan
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        return float(value[0]), float(value[1])
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            parsed = ast.literal_eval(value)
        if isinstance(parsed, (list, tuple)) and len(parsed) >= 2:
            return float(parsed[0]), float(parsed[1])
    return np.nan, np.nan


def appearance_info_from_lineups(positions_list, match_end_time):
    playing_time = []
    by_position = Counter()
    started = False

    for position in positions_list or []:
        start = _clock_to_seconds(position.get("from"))
        if position.get("start_reason") == "Starting XI":
            started = True
        end = _clock_to_seconds(position.get("to"))
        if end is None:
            end = match_end_time
        playing_time.append((start, end))
        pos_name = str(position.get("position") or "").strip()
        if pos_name and start is not None and end is not None and end > start:
            by_position[pos_name] += (end - start) / 60.0

    if not playing_time:
        return 0.0, False, by_position

    playing_time.sort(key=lambda pair: (pair[0] is None, pair[0] or 0.0))
    merged = []
    for start, end in playing_time:
        if start is None or end is None or end <= start:
            continue
        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(merged[-1][1], end)
    minutes = sum(end - start for start, end in merged) / 60.0
    return minutes, started, by_position


def parse_positions_cell(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return ast.literal_eval(value)
    return []


def _xy_arrays(series: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    """Parse location-like cells into float arrays without building a Series of tuples."""
    n = len(series)
    xs = np.full(n, np.nan, dtype=np.float64)
    ys = np.full(n, np.nan, dtype=np.float64)
    for i, value in enumerate(series.to_numpy()):
        x, y = parse_xy(value)
        xs[i] = x
        ys[i] = y
    return xs, ys


def prepare_events(frame: pd.DataFrame) -> pd.DataFrame:
    # Mutate in place: a deep copy of the object block alone needs ~1.2 GiB.
    out = frame
    out["player_id"] = pd.to_numeric(out["player_id"], errors="coerce")
    out["team_id"] = pd.to_numeric(out["team_id"], errors="coerce")
    out["possession_team_id"] = pd.to_numeric(out["possession_team_id"], errors="coerce")
    out["event_time"] = out["minute"].to_numpy(dtype=float) * 60.0 + out["second"].to_numpy(dtype=float)

    out["x"], out["y"] = _xy_arrays(out["location"])
    out["carry_end_x"], out["carry_end_y"] = _xy_arrays(out["carry_end_location"])
    out["pass_end_x"], out["pass_end_y"] = _xy_arrays(out["pass_end_location"])
    out.drop(columns=["location", "carry_end_location", "pass_end_location"], inplace=True)

    out["under_pressure"] = out["under_pressure"].fillna(False).astype(bool)
    out["counterpress"] = out["counterpress"].fillna(False).astype(bool)
    for col in [
        "pass_outcome",
        "dribble_outcome",
        "duel_type",
        "duel_outcome",
        "interception_outcome",
        "type",
    ]:
        out[col] = out[col].astype("category")
    return out


def build_lineup_minutes(lineups_frame: pd.DataFrame, events_frame: pd.DataFrame) -> pd.DataFrame:
    match_end = (
        events_frame.groupby("match_id", as_index=False)["event_time"]
        .max()
        .rename(columns={"event_time": "match_end_time"})
    )
    rows = []
    merged = lineups_frame.merge(match_end, on="match_id", how="left")
    for row in merged.itertuples(index=False):
        positions_list = parse_positions_cell(row.positions)
        minutes, started, pos_minutes = appearance_info_from_lineups(
            positions_list, float(row.match_end_time or 0.0)
        )
        if minutes <= 0:
            continue
        rows.append(
            {
                "match_id": int(row.match_id),
                "competition_id": int(row.competition_id),
                "season_id": int(row.season_id),
                "player_id": int(row.player_id),
                "player_name": row.player_name,
                "country": row.country,
                "team_name": row.team_name,
                "minutes": minutes,
                "started": int(started),
                "pos_minutes": dict(pos_minutes),
            }
        )
    return pd.DataFrame(rows)


# 7. Volume metrics

Season counts used by the sheet panels and the style / quality feature blocks.


In [6]:
import gc

ev = prepare_events(events)
del events
gc.collect()

player_match = build_lineup_minutes(lineups, ev)
print(f"player-match rows with minutes: {len(player_match):,}")

player_name = {}
country = {}
teams_seen = defaultdict(set)
minutes_by_position = defaultdict(Counter)
season_base = defaultdict(lambda: {"appearances": 0, "starts": 0, "minutes": 0.0})

for row in player_match.itertuples(index=False):
    key = (row.competition_id, row.season_id, row.player_id)
    player_name[key] = row.player_name
    country[key] = row.country
    teams_seen[key].add(row.team_name)
    minutes_by_position[key].update(row.pos_minutes)
    season_base[key]["appearances"] += 1
    season_base[key]["starts"] += int(row.started)
    season_base[key]["minutes"] += float(row.minutes)

appeared_frame = player_match[["match_id", "player_id"]].drop_duplicates()

is_pass = ev["type"] == "Pass"
is_carry = ev["type"] == "Carry"
is_dribble = ev["type"] == "Dribble"
is_pressure = ev["type"] == "Pressure"
is_interception = ev["type"] == "Interception"
is_block = ev["type"] == "Block"
is_recovery = ev["type"] == "Ball Recovery"
is_tackle = (ev["type"] == "Duel") & (ev["duel_type"] == "Tackle")
is_dribbled_past = ev["type"] == "Dribbled Past"
pass_complete = is_pass & ev["pass_outcome"].isna()
dribble_complete = is_dribble & (ev["dribble_outcome"] == "Complete")
tackle_won = is_tackle & ev["duel_outcome"].isin(["Won", "Success In Play", "Success Out"])
interception_won = is_interception & ev["interception_outcome"].isin(
    ["Won", "Success In Play", "Success Out"]
)

carry_dist = np.hypot(ev["carry_end_x"] - ev["x"], ev["carry_end_y"] - ev["y"])
long_carry = is_carry & carry_dist.ge(8.0)
prog_pass = pass_complete & (
    ((ev["pass_end_x"] - ev["x"]) >= 10.0) | ((ev["x"] < 80) & (ev["pass_end_x"] >= 80))
)
prog_carry = is_carry & (
    ((ev["carry_end_x"] - ev["x"]) >= 10.0) | ((ev["x"] < 80) & (ev["carry_end_x"] >= 80))
)

on_ball_under_pressure = (is_pass | is_dribble | is_carry) & ev["under_pressure"]
on_ball_keep_success = (
    (pass_complete & ev["under_pressure"])
    | (dribble_complete & ev["under_pressure"])
    | (is_carry & ev["under_pressure"] & carry_dist.fillna(0).ge(1.0))
)

# Assign in place instead of DataFrame.assign (avoids another full copy).
flagged = ev
flagged["passes"] = is_pass
flagged["passes_completed"] = pass_complete
flagged["carries"] = is_carry
flagged["long_carries"] = long_carry
flagged["dribbles"] = is_dribble
flagged["dribbles_completed"] = dribble_complete
flagged["pressures"] = is_pressure
flagged["counterpresses"] = flagged["counterpress"]
flagged["tackles"] = is_tackle
flagged["tackles_won"] = tackle_won
flagged["interceptions"] = is_interception
flagged["interceptions_won"] = interception_won
flagged["blocks"] = is_block
flagged["ball_recoveries"] = is_recovery
flagged["dribbled_past"] = is_dribbled_past
flagged["engagements"] = is_pressure | is_tackle | is_interception | is_block
flagged["direct_regain_events"] = tackle_won | interception_won | is_recovery
flagged["under_pressure_actions"] = on_ball_under_pressure
flagged["under_pressure_kept"] = on_ball_keep_success
flagged["miscontrols"] = flagged["type"] == "Miscontrol"
flagged["dispossessed"] = flagged["type"] == "Dispossessed"
flagged["turnovers"] = (
    flagged["type"].isin(["Miscontrol", "Dispossessed"])
    | (is_dribble & (flagged["dribble_outcome"] != "Complete"))
    | (is_pass & flagged["pass_outcome"].notna())
)
flagged["build_up_touches"] = flagged["player_id"].notna() & flagged["x"].lt(40)
flagged["finish_touches"] = flagged["player_id"].notna() & flagged["x"].ge(80)
flagged["progressive_passes"] = prog_pass
flagged["progressive_carries"] = prog_carry
flagged["carry_duration"] = np.where(is_carry, flagged["duration"].fillna(0.0), 0.0)
flagged["is_engagement"] = flagged["engagements"]
flagged["is_direct_regain"] = flagged["direct_regain_events"]
flagged["is_prog_next"] = prog_pass | prog_carry
flagged.drop(
    columns=[
        "minute",
        "second",
        "y",
        "carry_end_x",
        "carry_end_y",
        "pass_end_x",
        "pass_end_y",
        "duel_type",
        "duel_outcome",
        "interception_outcome",
        "counterpress",
        "duration",
    ],
    inplace=True,
    errors="ignore",
)
gc.collect()

COUNT_COLS = [
    "passes",
    "passes_completed",
    "carries",
    "long_carries",
    "dribbles",
    "dribbles_completed",
    "pressures",
    "counterpresses",
    "tackles",
    "tackles_won",
    "interceptions",
    "interceptions_won",
    "blocks",
    "ball_recoveries",
    "dribbled_past",
    "engagements",
    "direct_regain_events",
    "under_pressure_actions",
    "under_pressure_kept",
    "miscontrols",
    "dispossessed",
    "turnovers",
    "build_up_touches",
    "finish_touches",
    "progressive_passes",
    "progressive_carries",
]

player_events = flagged.dropna(subset=["player_id"])
player_events = player_events.assign(player_id=player_events["player_id"].astype(int))
player_events = player_events.merge(appeared_frame, on=["match_id", "player_id"], how="inner")

volume = (
    player_events.groupby(["competition_id", "season_id", "player_id"], as_index=False)[COUNT_COLS]
    .sum()
)
carry_time = (
    player_events.groupby(["competition_id", "season_id", "player_id"], as_index=False)[
        "carry_duration"
    ]
    .sum()
    .rename(columns={"carry_duration": "carry_seconds"})
)
volume = volume.merge(carry_time, on=["competition_id", "season_id", "player_id"], how="left")
print(f"volume rows: {len(volume):,}")


player-match rows with minutes: 43,044
volume rows: 2,554


# 8. Sequence metrics

TIP / OTIP rates, possession retention, regains, give-and-gos, and the danger proxy.


In [7]:
from tqdm.auto import tqdm

# --- Team TIP / OTIP from consecutive event timestamps ------------------------
poss_time_rows = []
for match_id, match_events in tqdm(
    flagged[["match_id", "period", "event_time", "possession_team_id", "competition_id", "season_id"]]
    .groupby("match_id", sort=False),
    total=flagged["match_id"].nunique(),
    desc="TIP/OTIP",
):
    match_events = match_events.sort_values(["period", "event_time"])
    dt = match_events["event_time"].diff()
    same_period = match_events["period"].eq(match_events["period"].shift())
    valid = same_period & dt.gt(0) & dt.le(120)
    tip = (
        match_events.loc[valid]
        .assign(dt=dt[valid])
        .groupby("possession_team_id", as_index=False)["dt"]
        .sum()
        .rename(columns={"possession_team_id": "team_id", "dt": "tip_seconds"})
    )
    tip["match_id"] = match_id
    tip["competition_id"] = int(match_events["competition_id"].iloc[0])
    tip["season_id"] = int(match_events["season_id"].iloc[0])
    tip["match_tip_seconds"] = tip["tip_seconds"].sum()
    poss_time_rows.append(tip)

team_tip = pd.concat(poss_time_rows, ignore_index=True)
team_tip["otip_seconds"] = team_tip["match_tip_seconds"] - team_tip["tip_seconds"]

player_team = (
    flagged.dropna(subset=["player_id", "team_id"])
    .groupby(["match_id", "player_id"], as_index=False)["team_id"]
    .agg(lambda s: s.mode().iloc[0])
)
player_team["player_id"] = player_team["player_id"].astype(int)
player_team["team_id"] = player_team["team_id"].astype(int)

player_poss_time = player_team.merge(team_tip, on=["match_id", "team_id"], how="left")
poss_time = (
    player_poss_time.groupby(["competition_id", "season_id", "player_id"], as_index=False)[
        ["tip_seconds", "otip_seconds"]
    ]
    .sum()
)

# --- Possession involvement / retention / build-up / finish -------------------
poss = flagged.dropna(subset=["possession"]).copy()
poss["poss_key"] = (
    poss["match_id"].astype(str) + "-" + poss["possession"].astype(str)
)
poss_meta = (
    poss.groupby("poss_key", as_index=False)
    .agg(
        competition_id=("competition_id", "first"),
        season_id=("season_id", "first"),
        start_x=("x", "first"),
        end_x=("x", "last"),
    )
)
player_poss = (
    poss.dropna(subset=["player_id"])
    .loc[:, ["poss_key", "competition_id", "season_id", "player_id"]]
    .drop_duplicates()
)
player_poss["player_id"] = player_poss["player_id"].astype(int)
player_poss = player_poss.merge(
    poss_meta[["poss_key", "start_x", "end_x"]], on="poss_key", how="left"
)

# Player-caused turnover inside a possession
turnover_mask = (
    poss["type"].isin(["Miscontrol", "Dispossessed"])
    | ((poss["type"] == "Pass") & poss["pass_outcome"].notna())
    | (
        (poss["type"] == "Dribble")
        & poss["dribble_outcome"].notna()
        & (poss["dribble_outcome"] != "Complete")
    )
)
player_turnovers = (
    poss.loc[turnover_mask & poss["player_id"].notna(), ["poss_key", "player_id"]]
    .drop_duplicates()
    .assign(player_id=lambda d: d["player_id"].astype(int), lost=True)
)
player_poss = player_poss.merge(player_turnovers, on=["poss_key", "player_id"], how="left")
player_poss["lost"] = player_poss["lost"].fillna(False)
player_poss["build_up_possessions"] = player_poss["start_x"].lt(40)
player_poss["finish_possessions"] = player_poss["end_x"].ge(80)
player_poss["possessions_retained"] = ~player_poss["lost"]

poss_stats = (
    player_poss.groupby(["competition_id", "season_id", "player_id"], as_index=False)
    .agg(
        possessions_involved=("poss_key", "count"),
        possessions_retained=("possessions_retained", "sum"),
        build_up_possessions=("build_up_possessions", "sum"),
        finish_possessions=("finish_possessions", "sum"),
    )
)

# --- Ordered-window metrics per match ----------------------------------------
window_acc = defaultdict(
    lambda: {
        "indirect_regains": 0,
        "danger_stopped": 0,
        "danger_reduced": 0,
        "danger_failed": 0,
        "give_and_gos": 0,
        "draws_pressure": 0,
        "dangerous_under_pressure": 0,
    }
)

need_cols = [
    "match_id",
    "competition_id",
    "season_id",
    "period",
    "event_time",
    "index",
    "type",
    "player_id",
    "team_id",
    "possession_team_id",
    "under_pressure",
    "x",
    "passes_completed",
    "is_engagement",
    "is_direct_regain",
    "is_prog_next",
]

for match_id, match_events in tqdm(
    flagged[need_cols].groupby("match_id", sort=False),
    total=flagged["match_id"].nunique(),
    desc="Windows",
):
    m = match_events.sort_values(["period", "event_time", "index"]).reset_index(drop=True)
    times = m["event_time"].to_numpy()
    types = m["type"].to_numpy()
    pids = m["player_id"].to_numpy()
    teams = m["team_id"].to_numpy()
    poss_teams = m["possession_team_id"].to_numpy()
    under_p = m["under_pressure"].to_numpy()
    xs = m["x"].to_numpy()
    pass_ok = m["passes_completed"].to_numpy()
    is_eng = m["is_engagement"].to_numpy()
    is_dir = m["is_direct_regain"].to_numpy()
    is_prog = m["is_prog_next"].to_numpy()
    competition_id = int(m["competition_id"].iloc[0])
    season_id = int(m["season_id"].iloc[0])
    n = len(m)

    for i in range(n):
        pid = pids[i]
        if pid != pid:
            continue
        pid_i = int(pid)
        key = (competition_id, season_id, pid_i)
        t0 = times[i]
        typ = types[i]

        if typ == "Pass" and bool(pass_ok[i]):
            j = i + 1
            while j < n and times[j] - t0 <= 2.0:
                if types[j] == "Ball Receipt*" and pids[j] == pid:
                    window_acc[key]["give_and_gos"] += 1
                    break
                j += 1

        if typ in {"Pass", "Carry", "Ball Receipt*", "Dribble"} and not bool(under_p[i]):
            j = i + 1
            while j < n and times[j] - t0 <= 3.0:
                if (
                    pids[j] == pid
                    and bool(under_p[j])
                    and types[j] in {"Pass", "Carry", "Dribble"}
                ):
                    window_acc[key]["draws_pressure"] += 1
                    break
                j += 1

        if bool(under_p[i]) and typ in {"Pass", "Carry", "Dribble", "Ball Receipt*"}:
            j = i
            while j < n and times[j] - t0 <= 5.0:
                if pids[j] == pid and (bool(is_prog[j]) or types[j] == "Shot"):
                    window_acc[key]["dangerous_under_pressure"] += 1
                    break
                j += 1

        if not bool(is_eng[i]):
            continue

        my_team = teams[i]
        stopped = False
        reduced = False
        j = i + 1
        while j < n and times[j] - t0 <= 5.0:
            if poss_teams[j] == my_team and poss_teams[i] != my_team:
                stopped = True
                if (not bool(is_dir[i])) and pids[j] != pid:
                    window_acc[key]["indirect_regains"] += 1
                break
            if poss_teams[j] != my_team and bool(is_prog[j]):
                break
            if (
                poss_teams[j] != my_team
                and types[j] in {"Pass", "Carry"}
                and xs[i] == xs[i]
                and xs[j] == xs[j]
                and xs[j] <= xs[i] - 5.0
            ):
                reduced = True
                break
            j += 1

        if stopped:
            window_acc[key]["danger_stopped"] += 1
        elif reduced:
            window_acc[key]["danger_reduced"] += 1
        else:
            window_acc[key]["danger_failed"] += 1

window_stats = pd.DataFrame(
    [
        {"competition_id": k[0], "season_id": k[1], "player_id": k[2], **v}
        for k, v in window_acc.items()
    ]
)

seq = poss_time.merge(poss_stats, on=["competition_id", "season_id", "player_id"], how="outer")
seq = seq.merge(window_stats, on=["competition_id", "season_id", "player_id"], how="outer")
print(f"sequence rows: {len(seq):,}")


TIP/OTIP:   0%|          | 0/1551 [00:00<?, ?it/s]

Windows:   0%|          | 0/1551 [00:00<?, ?it/s]

sequence rows: 2,556


# 9. Assemble player-season table

Join identity, volume, and sequence fields; write `player_metrics.parquet`.


In [8]:
rows = []
for key, base in season_base.items():
    competition_id, season_id, player_id = key
    pos_minutes = minutes_by_position[key]
    main_position = pos_minutes.most_common(1)[0][0] if pos_minutes else ""
    rows.append(
        {
            "competition_id": competition_id,
            "season_id": season_id,
            "player_id": player_id,
            "player_name": player_name.get(key, ""),
            "country": country.get(key, ""),
            "teams": ", ".join(sorted(teams_seen[key])),
            "main_position": main_position,
            "positions": ", ".join(sorted(pos_minutes.keys())),
            "appearances": base["appearances"],
            "starts": base["starts"],
            "minutes": round(base["minutes"], 1),
        }
    )

players = pd.DataFrame(rows)
players = players.merge(volume, on=["competition_id", "season_id", "player_id"], how="left")
players = players.merge(seq, on=["competition_id", "season_id", "player_id"], how="left")

comp_lookup = matches[
    ["competition_id", "season_id", "competition_name", "season_name"]
].drop_duplicates()
players = players.merge(comp_lookup, on=["competition_id", "season_id"], how="left")

id_cols = {
    "competition_id",
    "season_id",
    "player_id",
    "player_name",
    "country",
    "teams",
    "main_position",
    "positions",
    "competition_name",
    "season_name",
}
num_cols = [col for col in players.columns if col not in id_cols]
players[num_cols] = players[num_cols].fillna(0)

minutes = players["minutes"].replace(0, np.nan)
tip30 = (players["tip_seconds"] / 60.0 / 30.0).replace(0, np.nan)
otip30 = (players["otip_seconds"] / 60.0 / 30.0).replace(0, np.nan)

players["passes_p90"] = (players["passes"] * 90.0 / minutes).round(3)
players["pressures_p90"] = (players["pressures"] * 90.0 / minutes).round(3)
players["engagements_p90"] = (players["engagements"] * 90.0 / minutes).round(3)
players["long_carries_p90"] = (players["long_carries"] * 90.0 / minutes).round(3)
players["progressive_passes_p90"] = (players["progressive_passes"] * 90.0 / minutes).round(3)
players["progressive_carries_p90"] = (players["progressive_carries"] * 90.0 / minutes).round(3)

players["passes_per_30_tip"] = (players["passes"] / tip30).round(3)
players["pressures_per_30_otip"] = (players["pressures"] / otip30).round(3)
players["long_carries_per_30_tip"] = (players["long_carries"] / tip30).round(3)
players["give_and_gos_per_30_tip"] = (players["give_and_gos"] / tip30).round(3)
players["draws_pressure_per_30_tip"] = (players["draws_pressure"] / tip30).round(3)

players["pass_completion"] = (
    players["passes_completed"] / players["passes"].replace(0, np.nan)
).fillna(0).round(3)
players["possessions_retained_pct"] = (
    players["possessions_retained"] / players["possessions_involved"].replace(0, np.nan)
).fillna(0).round(3)
players["keep_under_pressure_pct"] = (
    players["under_pressure_kept"] / players["under_pressure_actions"].replace(0, np.nan)
).fillna(0).round(3)
players["direct_regain_pct"] = (
    players["direct_regain_events"] / players["engagements"].replace(0, np.nan)
).fillna(0).round(3)
players["indirect_regain_pct"] = (
    players["indirect_regains"] / players["engagements"].replace(0, np.nan)
).fillna(0).round(3)
players["regain_pct"] = (
    (players["direct_regain_events"] + players["indirect_regains"])
    / players["engagements"].replace(0, np.nan)
).fillna(0).round(3)
players["win_ball_when_engaging_pct"] = players["direct_regain_pct"]
players["danger_stopped_or_reduced_pct"] = (
    (players["danger_stopped"] + players["danger_reduced"])
    / (
        players["danger_stopped"]
        + players["danger_reduced"]
        + players["danger_failed"]
    ).replace(0, np.nan)
).fillna(0).round(3)
players["avg_possession_length_s"] = (
    players["carry_seconds"] / players["carries"].replace(0, np.nan)
).fillna(0).round(3)
players["build_up_possessions_p90"] = (
    players["build_up_possessions"] * 90.0 / minutes
).round(3)
players["finish_possessions_p90"] = (
    players["finish_possessions"] * 90.0 / minutes
).round(3)
players["dangerous_under_pressure_p90"] = (
    players["dangerous_under_pressure"] * 90.0 / minutes
).round(3)

players = players.sort_values(
    ["competition_name", "minutes", "player_name"], ascending=[True, False, True]
).reset_index(drop=True)

players.to_parquet(PLAYER_METRICS_PATH, index=False)
print(f"Wrote {PLAYER_METRICS_PATH.relative_to(ROOT)} ({len(players):,} players)")
display(
    players[
        [
            "player_name",
            "competition_name",
            "main_position",
            "minutes",
            "engagements_p90",
            "pressures_per_30_otip",
            "regain_pct",
            "possessions_retained_pct",
            "long_carries_p90",
        ]
    ].head(10)
)


Wrote projects\data\kante-replacement\player_metrics.parquet (2,560 players)


,player_name,competition_name,main_position,minutes,engagements_p90,pressures_per_30_otip,regain_pct,possessions_retained_pct,long_carries_p90
0,Bernd Leno,1. Bundesliga,Goalkeeper,3079.6,0.292,0.164,18.000,0.709,4.325
1,Karim Bellarabi,1. Bundesliga,Right Midfield,2661.5,25.970,13.045,0.340,0.707,11.700
2,Jonathan Tah,1. Bundesliga,Right Center Back,2659.5,15.567,7.264,0.667,0.820,7.986
3,Wendell Nascimento Borges,1. Bundesliga,Left Back,2498.7,24.781,11.137,0.442,0.748,9.329
4,Christoph Kramer,1. Bundesliga,Right Defensive Midfield,2477.1,28.340,15.464,0.376,0.798,10.755
5,Hakan Çalhanoğlu,1. Bundesliga,Left Midfield,2314.9,22.822,10.425,0.359,0.697,9.292
6,Javier Hernández Balcázar,1. Bundesliga,Center Forward,2219.6,11.921,6.302,0.272,0.815,3.974
7,Stefan Kießling,1. Bundesliga,Left Center Forward,1971.5,21.136,8.764,0.272,0.766,3.241
8,Kevin Kampl,1. Bundesliga,Left Defensive Midfield,1806.3,25.561,12.317,0.481,0.800,14.101
9,Julian Brandt,1. Bundesliga,Left Midfield,1678.8,21.766,8.216,0.315,0.753,12.062


# 10. Filter midfielders

Keep midfield labels with 900+ minutes. Write `midfielders.parquet` (the 413-player pool on the slides).


In [9]:
midfielders = players[
    players["main_position"].isin(MIDFIELD_LABELS) & players["minutes"].ge(MINUTES_FLOOR)
].copy()

midfielders = midfielders.sort_values(
    ["engagements_p90", "minutes"], ascending=[False, False]
).reset_index(drop=True)

midfielders.to_parquet(MIDFIELDERS_PATH, index=False)
print(
    f"Midfielders with >= {MINUTES_FLOOR} minutes: {len(midfielders):,} "
    f"(from {players['main_position'].isin(MIDFIELD_LABELS).sum():,} midfield rows)"
)
print(f"Wrote {MIDFIELDERS_PATH.relative_to(ROOT)}")

display(
    midfielders.groupby(["competition_name", "main_position"], as_index=False)
    .size()
    .sort_values(["competition_name", "size"], ascending=[True, False])
    .head(40)
)

show_cols = [
    "player_name",
    "competition_name",
    "main_position",
    "teams",
    "minutes",
    "engagements_p90",
    "pressures_per_30_otip",
    "regain_pct",
    "direct_regain_pct",
    "danger_stopped_or_reduced_pct",
    "possessions_retained_pct",
    "keep_under_pressure_pct",
    "long_carries_p90",
    "build_up_possessions_p90",
    "finish_possessions_p90",
    "passes_per_30_tip",
    "give_and_gos_per_30_tip",
    "avg_possession_length_s",
]
display(midfielders[show_cols].head(20))

# Quick Kanté check if present in open PL 2015/16.
kante = midfielders[midfielders["player_name"].str.contains("Kant", case=False, na=False)]
display(kante[show_cols] if len(kante) else "Kanté not in midfielder filter")


Midfielders with >= 900 minutes: 413 (from 822 midfield rows)
Wrote projects\data\kante-replacement\midfielders.parquet


,competition_name,main_position,size
1,1. Bundesliga,Left Midfield,3
2,1. Bundesliga,Right Defensive Midfield,2
0,1. Bundesliga,Left Defensive Midfield,1
3,1. Bundesliga,Right Midfield,1
10,La Liga,Right Defensive Midfield,23
7,La Liga,Left Defensive Midfield,21
4,La Liga,Center Attacking Midfield,16
8,La Liga,Left Midfield,11
11,La Liga,Right Midfield,10
5,La Liga,Center Defensive Midfield,8


,player_name,competition_name,main_position,teams,minutes,engagements_p90,pressures_per_30_otip,regain_pct,direct_regain_pct,danger_stopped_or_reduced_pct,possessions_retained_pct,keep_under_pressure_pct,long_carries_p90,build_up_possessions_p90,finish_possessions_p90,passes_per_30_tip,give_and_gos_per_30_tip,avg_possession_length_s
0,Marco Crimi,Serie A,Right Center Midfield,"Bologna, Carpi",1074.5,50.591,12.914,0.240,0.189,0.339,0.797,0.765,5.109,32.331,17.506,13.621,0.374,1.497
1,Lorenzo Lollo,Serie A,Center Attacking Midfield,Carpi,2248.3,44.994,19.038,0.230,0.194,0.343,0.836,0.782,4.764,28.982,15.772,19.027,0.462,1.291
2,Vincent Koziello,Ligue 1,Right Center Midfield,OGC Nice,2768.2,40.868,20.546,0.278,0.206,0.355,0.871,0.819,7.673,31.244,15.866,30.140,1.732,1.317
3,Jean-Daniel Akpa Akpro,Ligue 1,Right Center Midfield,Toulouse,2655.0,38.881,16.717,0.305,0.252,0.321,0.772,0.771,8.475,28.102,16.407,22.924,0.876,1.422
4,Marten de Roon,Serie A,Center Defensive Midfield,Atalanta,3225.0,38.679,18.866,0.309,0.237,0.308,0.819,0.794,5.219,27.851,13.340,29.860,0.573,1.408
5,Mirko Gori,Serie A,Right Defensive Midfield,Frosinone,2306.3,37.970,15.668,0.334,0.285,0.243,0.799,0.772,6.790,25.209,17.053,24.778,0.611,1.558
6,Artur Ioniţă,Serie A,Right Defensive Midfield,Hellas Verona,2474.8,37.785,16.924,0.256,0.210,0.358,0.846,0.792,8.583,26.439,15.310,18.082,0.419,1.850
7,Nicolas Seube,Ligue 1,Center Defensive Midfield,"Caen, Stade Malherbe Caen",2240.7,37.314,12.563,0.301,0.221,0.301,0.863,0.782,4.458,27.393,13.697,17.812,0.284,1.514
8,Renaud Cohade,Ligue 1,Left Center Midfield,Saint-Étienne,1088.2,37.052,12.395,0.321,0.270,0.366,0.736,0.735,10.007,32.917,15.714,25.254,1.037,1.396
9,Idrissa Gana Gueye,Premier League,Left Center Midfield,Aston Villa,3202.8,37.008,17.222,0.395,0.348,0.295,0.817,0.846,13.348,25.122,17.338,35.455,1.488,1.638


,player_name,competition_name,main_position,teams,minutes,engagements_p90,pressures_per_30_otip,regain_pct,direct_regain_pct,danger_stopped_or_reduced_pct,possessions_retained_pct,keep_under_pressure_pct,long_carries_p90,build_up_possessions_p90,finish_possessions_p90,passes_per_30_tip,give_and_gos_per_30_tip,avg_possession_length_s
11,N''Golo Kanté,Premier League,Left Defensive Midfield,Leicester City,3179.5,36.628,14.022,0.423,0.371,0.301,0.826,0.839,9.907,24.881,14.351,27.263,0.628,1.606


# 11. Similarity model (Twin Test)

16 style percentiles and 9 quality percentiles against the midfielder pool. Euclidean distance to Kanté on each block, blended 60% style / 40% quality. Write `kante_twins.parquet`.


In [10]:
# Twin Test-style similarity: match style shape first, then quality level.
# Features are percentiles in the midfielder pool. Lower-is-better metrics inverted.

from numpy.linalg import norm

STYLE_FEATURES = [
    ("engagements_p90", "Engagements"),
    ("tackles_p90", "Tackles"),
    ("interceptions_p90", "Interceptions"),
    ("pressures_per_30_otip", "Pressures /30 OTIP"),
    ("pressures_p90", "Pressures"),
    ("counterpresses_p90", "Counterpresses"),
    ("blocks_p90", "Blocks"),
    ("dribbled_past_p90", "Dribbled past"),  # inverted
    ("progressive_passes_p90", "Prog. passes"),
    ("progressive_carries_p90", "Prog. carries"),
    ("long_carries_p90", "Long carries"),
    ("build_up_possessions_p90", "Build-up poss."),
    ("finish_possessions_p90", "Finish poss."),
    ("passes_per_30_tip", "Passes /30 TIP"),
    ("give_and_gos_per_30_tip", "Give-and-gos"),
    ("draws_pressure_per_30_tip", "Draws pressure"),
]

QUALITY_FEATURES = [
    ("regain_pct", "Regain%"),
    ("win_ball_when_engaging_pct", "Win when engaging%"),
    ("direct_regain_pct", "Direct regain%"),
    ("danger_stopped_or_reduced_pct", "Danger stop/reduce%"),
    ("possessions_retained_pct", "Poss. retained%"),
    ("keep_under_pressure_pct", "Keep under press%"),
    ("pass_completion", "Pass completion%"),
    ("turnovers_p90", "Turnovers"),  # inverted
    ("dangerous_under_pressure_p90", "Danger under press"),
]

INVERT = {"dribbled_past_p90", "turnovers_p90"}

# Twin Test-inspired weights: style dominates shortlist shape, quality breaks ties.
W_STYLE = 0.60
W_QUALITY = 0.40
TOP_N = 15

# Optional: mirror SkillCorner age band later. No DOB in open lineups, so skip.
DM_ONLY = False
DM_LABELS = {
    "Center Defensive Midfield",
    "Left Defensive Midfield",
    "Right Defensive Midfield",
}


def _ensure_p90(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    minutes = out["minutes"].replace(0, np.nan)
    for col in ["tackles", "interceptions", "blocks", "dribbled_past", "turnovers", "counterpresses"]:
        p90 = f"{col}_p90"
        if p90 not in out.columns:
            out[p90] = (out[col] * 90.0 / minutes).astype(float)
    return out


def _percentile_matrix(frame: pd.DataFrame, specs: list[tuple[str, str]]) -> pd.DataFrame:
    cols = {}
    for col, _label in specs:
        # Inverted metrics: low raw value => high percentile (ascending=False).
        ascending = col not in INVERT
        cols[col] = frame[col].rank(pct=True, ascending=ascending, method="average") * 100
    return pd.DataFrame(cols, index=frame.index)


def _block_distance(a: np.ndarray, b: np.ndarray) -> float:
    # Euclidean distance on percentile vectors scaled to [0, 1].
    return float(norm((a - b) / 100.0))


pool = _ensure_p90(midfielders)
if DM_ONLY:
    pool = pool[pool["main_position"].isin(DM_LABELS)].copy()

style_pct = _percentile_matrix(pool, STYLE_FEATURES)
quality_pct = _percentile_matrix(pool, QUALITY_FEATURES)

kante_mask = pool["player_name"].str.contains("Kant", case=False, na=False)
if kante_mask.sum() != 1:
    raise ValueError(f"Expected one Kanté row, found {kante_mask.sum()}")
kante_idx = pool.index[kante_mask][0]
kante_row = pool.loc[kante_idx]

kante_style = style_pct.loc[kante_idx].to_numpy(dtype=float)
kante_quality = quality_pct.loc[kante_idx].to_numpy(dtype=float)

# Max possible Euclidean distance on unit-scaled vectors for normalisation.
style_dim = len(STYLE_FEATURES)
quality_dim = len(QUALITY_FEATURES)
style_max = float(np.sqrt(style_dim))
quality_max = float(np.sqrt(quality_dim))

rows = []
for idx in pool.index:
    if idx == kante_idx:
        continue
    row = pool.loc[idx]
    d_style = _block_distance(style_pct.loc[idx].to_numpy(dtype=float), kante_style)
    d_quality = _block_distance(quality_pct.loc[idx].to_numpy(dtype=float), kante_quality)
    d_total = W_STYLE * (d_style / style_max) + W_QUALITY * (d_quality / quality_max)
    similarity = 100.0 * (1.0 - d_total)
    rows.append(
        {
            "player_id": int(row["player_id"]),
            "player_name": row["player_name"],
            "competition_name": row["competition_name"],
            "teams": row["teams"],
            "main_position": row["main_position"],
            "minutes": float(row["minutes"]),
            "style_distance": round(d_style, 4),
            "quality_distance": round(d_quality, 4),
            "style_similarity": round(100.0 * (1.0 - d_style / style_max), 2),
            "quality_similarity": round(100.0 * (1.0 - d_quality / quality_max), 2),
            "similarity": round(similarity, 2),
        }
    )

twins = (
    pd.DataFrame(rows)
    .sort_values(["similarity", "style_similarity", "quality_similarity"], ascending=False)
    .reset_index(drop=True)
)
twins.insert(0, "rank", twins.index + 1)

# SkillCorner-style output: top 3 "closest" twins, plus a wider shortlist.
top3 = twins.head(3).copy()
top_n = twins.head(TOP_N).copy()

TWINS_PATH = DATA_DIR / "kante_twins.parquet"
TOPN_PATH = DATA_DIR / "kante_twins_top15.csv"
twins.to_parquet(TWINS_PATH, index=False)
top_n.to_csv(TOPN_PATH, index=False)

print(
    f"Pool: {len(pool):,} midfielders | "
    f"style dims={style_dim} | quality dims={quality_dim} | "
    f"weights style={W_STYLE:.0%} quality={W_QUALITY:.0%}"
)
print(
    f"Target: {str(kante_row['player_name']).replace(chr(39)*2, chr(39))} | "
    f"{kante_row['teams']} | {kante_row['main_position']} | {kante_row['minutes']:.0f} mins"
)
print(f"Wrote {TWINS_PATH.relative_to(ROOT)}")
print(f"Wrote {TOPN_PATH.relative_to(ROOT)}")
print("\nTop 3 twins")
display(
    top3[
        [
            "rank",
            "player_name",
            "competition_name",
            "teams",
            "main_position",
            "minutes",
            "similarity",
            "style_similarity",
            "quality_similarity",
        ]
    ]
)
print(f"\nTop {TOP_N}")
display(
    top_n[
        [
            "rank",
            "player_name",
            "competition_name",
            "teams",
            "main_position",
            "minutes",
            "similarity",
            "style_similarity",
            "quality_similarity",
            "style_distance",
            "quality_distance",
        ]
    ]
)


Pool: 413 midfielders | style dims=16 | quality dims=9 | weights style=60% quality=40%
Target: N'Golo Kanté | Leicester City | Left Defensive Midfield | 3180 mins
Wrote projects\data\kante-replacement\kante_twins.parquet
Wrote projects\data\kante-replacement\kante_twins_top15.csv

Top 3 twins


,rank,player_name,competition_name,teams,main_position,minutes,similarity,style_similarity,quality_similarity
0,1,Jaba Kankava,Ligue 1,Stade de Reims,Left Defensive Midfield,2453.6,81.53,81.58,81.44
1,2,Jack Frank Porteous Cork,Premier League,Swansea City,Right Center Midfield,2675.8,81.37,76.90,88.07
2,3,Idrissa Gana Gueye,Premier League,Aston Villa,Left Center Midfield,3202.8,79.61,73.70,88.48



Top 15


,rank,player_name,competition_name,teams,main_position,minutes,similarity,style_similarity,quality_similarity,style_distance,quality_distance
0,1,Jaba Kankava,Ligue 1,Stade de Reims,Left Defensive Midfield,2453.6,81.53,81.58,81.44,0.7368,0.5567
1,2,Jack Frank Porteous Cork,Premier League,Swansea City,Right Center Midfield,2675.8,81.37,76.90,88.07,0.9240,0.3578
2,3,Idrissa Gana Gueye,Premier League,Aston Villa,Left Center Midfield,3202.8,79.61,73.70,88.48,1.0521,0.3457
3,4,Asier Illarramendi Andonegi,La Liga,Real Sociedad,Right Defensive Midfield,2942.3,79.31,80.35,77.74,0.7861,0.6677
4,5,Petros Matheus dos Santos Araújo,La Liga,Real Betis,Right Defensive Midfield,2462.0,78.58,85.20,68.65,0.5922,0.9404
5,6,Mehdi Lacen,La Liga,Getafe,Left Defensive Midfield,2750.8,78.27,70.40,90.06,1.1839,0.2982
6,7,Mousa Sidi Yaya Dembélé,Premier League,Tottenham Hotspur,Left Defensive Midfield,2346.4,77.94,74.43,83.21,1.0227,0.5037
7,8,Victor Wanyama,Premier League,Southampton,Right Defensive Midfield,2678.3,77.69,71.43,87.07,1.1427,0.3879
8,9,Augusto Matías Fernández,La Liga,"Atlético Madrid, Celta Vigo",Center Defensive Midfield,2256.5,77.49,75.45,80.54,0.9819,0.5837
9,10,José Luis García del Pozo,La Liga,Málaga,Left Defensive Midfield,2829.6,77.01,71.50,85.27,1.1400,0.4418


# 12. Numbers for the slides

Print the sheet panel values / percentiles, twin style and quality match scores, and the compare-page bars. Paste from here into `slides/kante-replacement/index.tsx` when the pool changes.


In [11]:
# Numbers used on slides/kante-replacement/index.tsx.
minutes = pool["minutes"].replace(0, np.nan)
for col in [
    "tackles",
    "interceptions",
    "blocks",
    "dribbled_past",
    "turnovers",
    "counterpresses",
    "ball_recoveries",
    "miscontrols",
    "dispossessed",
]:
    pool[f"{col}_p90"] = pool[col] * 90.0 / minutes

INVERT_SHEET = {
    "dribbled_past_p90",
    "turnovers_p90",
    "miscontrols_p90",
    "dispossessed_p90",
}

SHEET = {
    "Ball-winning": [
        "engagements_p90",
        "tackles_p90",
        "interceptions_p90",
        "ball_recoveries_p90",
        "regain_pct",
        "direct_regain_pct",
        "indirect_regain_pct",
    ],
    "Cover & disrupt": [
        "pressures_p90",
        "pressures_per_30_otip",
        "counterpresses_p90",
        "blocks_p90",
        "danger_stopped_or_reduced_pct",
        "dribbled_past_p90",
    ],
    "On-ball reliability": [
        "possessions_retained_pct",
        "keep_under_pressure_pct",
        "pass_completion",
        "turnovers_p90",
        "miscontrols_p90",
        "dispossessed_p90",
        "dangerous_under_pressure_p90",
    ],
    "Progression": [
        "progressive_passes_p90",
        "progressive_carries_p90",
        "long_carries_p90",
        "passes_per_30_tip",
        "build_up_possessions_p90",
        "finish_possessions_p90",
        "give_and_gos_per_30_tip",
    ],
}

COMPARE = [
    "engagements_p90",
    "tackles_p90",
    "interceptions_p90",
    "pressures_per_30_otip",
    "regain_pct",
    "keep_under_pressure_pct",
    "progressive_carries_p90",
    "progressive_passes_p90",
]


def _pct(col: str) -> pd.Series:
    return pool[col].rank(pct=True, ascending=col not in INVERT_SHEET, method="average") * 100


def _rank(col: str) -> pd.Series:
    return pool[col].rank(ascending=col in INVERT_SHEET, method="min")


kante_idx = pool.index[pool["player_name"].str.contains("Kant", na=False)][0]
print(f"pool={len(pool)}")
print(pool.loc[kante_idx, ["player_name", "teams", "minutes", "appearances", "starts", "positions"]].to_string())

print("\n=== Kanté sheet ===")
for panel, cols in SHEET.items():
    print(f"-- {panel}")
    for col in cols:
        print(
            f"   {col:32s} value={pool.loc[kante_idx, col]:.3f}"
            f"  pct={_pct(col).loc[kante_idx]:.0f}  rank={int(_rank(col).loc[kante_idx])}"
        )

print("\n=== Twins (style / quality match scores) ===")
twins_sorted = twins.sort_values("rank")
for _, t in twins_sorted.head(3).iterrows():
    idx = pool.index[pool["player_id"] == t["player_id"]][0]
    row = pool.loc[idx]
    print(
        f"#{int(t['rank'])} {row['player_name']} | {row['teams']} | "
        f"sim={t['similarity']:.1f} style={t['style_similarity']:.1f} quality={t['quality_similarity']:.1f}"
    )
    for col in COMPARE:
        print(
            f"   {col:28s} pct={_pct(col).loc[idx]:.0f}"
            f"  kante_pct={_pct(col).loc[kante_idx]:.0f}"
        )

print("\nsimilarity distribution:")
print(twins_sorted["similarity"].describe().round(2).to_string())
print("rank 4-10:")
print(twins_sorted.iloc[3:10][["rank", "player_name", "teams", "similarity"]].to_string(index=False))



pool=413
player_name                                        N''Golo Kanté
teams                                             Leicester City
minutes                                                   3179.5
appearances                                                   37
starts                                                        33
positions      Center Attacking Midfield, Left Center Forward...

=== Kanté sheet ===
-- Ball-winning
   engagements_p90                  value=36.628  pct=97  rank=12
   tackles_p90                      value=4.642  pct=100  rank=1
   interceptions_p90                value=3.114  pct=100  rank=3
   ball_recoveries_p90              value=8.492  pct=99  rank=7
   regain_pct                       value=0.423  pct=77  rank=93
   direct_regain_pct                value=0.371  pct=78  rank=90
   indirect_regain_pct              value=0.053  pct=51  rank=194
-- Cover & disrupt
   pressures_p90                    value=25.646  pct=89  rank=45
   pressures_per_30_oti

# 13. Export slide charts

Write title-free style / quality pizzas, the similarity scatter, and twin minutes bars into `slides/kante-replacement/assets/`. Titles and player names live on the slides.


In [12]:
# Title-free charts for slides/kante-replacement/assets/.
import ast
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mplsoccer import PyPizza

ASSETS_DIR = ROOT / "slides" / "kante-replacement" / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

INK, ACCENT, GOOD, BAD = "#22303c", "#1b6ca8", "#2e8b57", "#c1553b"
MUTED, PAPER, RULE, WELL = "#5F7380", "#f7f5f2", "#6B8294", "#ffffff"
STYLE_COLOR, STYLE_BLANK = GOOD, "#d8ebe0"
QUALITY_COLOR, QUALITY_BLANK = ACCENT, "#d4e4f0"

plt.rcParams.update(
    {
        "figure.facecolor": PAPER,
        "axes.facecolor": PAPER,
        "savefig.facecolor": PAPER,
        "text.color": INK,
        "axes.labelcolor": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.edgecolor": RULE,
        "font.family": "Arial",
    }
)

STYLE_SPECS = [
    ("engagements_p90", "Engagements"),
    ("tackles_p90", "Tackles"),
    ("interceptions_p90", "Intercepts"),
    ("pressures_per_30_otip", "Press rate"),
    ("pressures_p90", "Pressures"),
    ("counterpresses_p90", "Counterpress"),
    ("blocks_p90", "Blocks"),
    ("dribbled_past_p90", "Dribbled past"),
    ("progressive_passes_p90", "Prog passes"),
    ("progressive_carries_p90", "Prog carries"),
    ("long_carries_p90", "Long carries"),
    ("build_up_possessions_p90", "Build-up"),
    ("finish_possessions_p90", "Finishing"),
    ("passes_per_30_tip", "Pass rate"),
    ("give_and_gos_per_30_tip", "Give-and-go"),
    ("draws_pressure_per_30_tip", "Draws press"),
]
QUALITY_SPECS = [
    ("regain_pct", "Regain %"),
    ("win_ball_when_engaging_pct", "Win when engaging"),
    ("direct_regain_pct", "Direct regain"),
    ("danger_stopped_or_reduced_pct", "Danger stopped"),
    ("possessions_retained_pct", "Retained %"),
    ("keep_under_pressure_pct", "Keep under press"),
    ("pass_completion", "Pass completion"),
    ("turnovers_p90", "Turnovers"),
    ("dangerous_under_pressure_p90", "Danger under press"),
]
INVERT_CHART = {"dribbled_past_p90", "turnovers_p90"}


def _chart_p90(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    minutes = out["minutes"].replace(0, np.nan)
    for col in ["tackles", "interceptions", "blocks", "dribbled_past", "turnovers", "counterpresses"]:
        out[f"{col}_p90"] = out[col] * 90.0 / minutes
    return out


def _pct_frame(frame: pd.DataFrame, specs) -> pd.DataFrame:
    return pd.DataFrame(
        {
            col: frame[col].rank(pct=True, ascending=col not in INVERT_CHART, method="average") * 100
            for col, _ in specs
        },
        index=frame.index,
    )


def _baker_kwargs() -> dict:
    return {
        "background_color": PAPER,
        "straight_line_color": RULE,
        "straight_line_lw": 1,
        "last_circle_color": INK,
        "last_circle_lw": 1.4,
        "other_circle_color": RULE,
        "other_circle_lw": 0.8,
        "other_circle_ls": "-",
        "inner_circle_size": 8,
    }


def _text_kwargs(slice_color: str) -> dict:
    return {
        "kwargs_slices": {"edgecolor": PAPER, "linewidth": 1.2, "zorder": 2},
        "kwargs_params": {
            "color": INK,
            "fontsize": 10,
            "fontproperties": {"family": "Arial", "weight": "bold"},
            "va": "center",
        },
        "kwargs_values": {
            "color": WELL,
            "fontsize": 9,
            "fontproperties": {"family": "Arial", "weight": "bold"},
            "zorder": 3,
            "bbox": {
                "edgecolor": PAPER,
                "facecolor": slice_color,
                "boxstyle": "round,pad=0.2",
                "lw": 0,
            },
        },
    }


def _save(fig, name: str) -> None:
    path = ASSETS_DIR / name
    fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0.05, facecolor=PAPER)
    plt.close(fig)
    print(f"Wrote {path.relative_to(ROOT)}")


chart_pool = _chart_p90(pd.read_parquet(DATA_DIR / "midfielders.parquet"))
chart_twins = pd.read_parquet(DATA_DIR / "kante_twins.parquet").sort_values("rank")
kante_i = chart_pool.index[chart_pool["player_name"].str.contains("Kant", na=False)][0]
players = [("kante", kante_i)]
for _, twin in chart_twins.head(3).iterrows():
    players.append(
        (f"twin{int(twin['rank'])}", chart_pool.index[chart_pool["player_id"] == twin["player_id"]][0])
    )


def _pizza(specs, pct_frame, idx, color, blank, name):
    values = [float(pct_frame.loc[idx, col]) for col, _ in specs]
    baker = PyPizza(params=[label for _, label in specs], **_baker_kwargs())
    text_kw = _text_kwargs(color)
    text_kw["kwargs_params"]["fontsize"] = 11 if len(specs) > 10 else 12
    text_kw["kwargs_values"]["fontsize"] = 11
    fig, _ = baker.make_pizza(
        values,
        alt_text_values=[f"{v:.0f}" for v in values],
        figsize=(5.8, 5.8),
        param_location=111,
        slice_colors=[color] * len(values),
        value_colors=[WELL] * len(values),
        value_bck_colors=[color] * len(values),
        color_blank_space=[blank] * len(values),
        blank_alpha=0.95,
        **text_kw,
    )
    _save(fig, name)


style_pct = _pct_frame(chart_pool, STYLE_SPECS)
quality_pct = _pct_frame(chart_pool, QUALITY_SPECS)
for key, idx in players:
    _pizza(STYLE_SPECS, style_pct, idx, STYLE_COLOR, STYLE_BLANK, f"style-{key}.png")
    _pizza(QUALITY_SPECS, quality_pct, idx, QUALITY_COLOR, QUALITY_BLANK, f"quality-{key}.png")

BAND = 5.0
sc = chart_twins.copy()
sc["delta"] = sc["quality_similarity"] - sc["style_similarity"]
above = sc["delta"] > BAND
below = sc["delta"] < -BAND
balanced = ~(above | below)
fig, ax = plt.subplots(figsize=(7.4, 7.0))
lim0, lim1 = 35, 100
xs = np.linspace(lim0, lim1, 50)
ax.fill_between(xs, xs - BAND, xs + BAND, color=RULE, alpha=0.18, linewidth=0, zorder=1)
ax.plot([lim0, lim1], [lim0, lim1], color=RULE, ls="--", lw=1.4, zorder=2)
for mask, color, alpha in [(above, BAD, 0.55), (below, GOOD, 0.55), (balanced, MUTED, 0.45)]:
    ax.scatter(
        sc.loc[mask, "style_similarity"],
        sc.loc[mask, "quality_similarity"],
        s=30,
        c=color,
        alpha=alpha,
        edgecolors="none",
        zorder=3,
    )
top3 = sc.head(3)
ax.scatter(
    top3["style_similarity"],
    top3["quality_similarity"],
    s=110,
    facecolors=WELL,
    edgecolors=INK,
    linewidths=2,
    zorder=5,
)
offsets = {1: (10, -16), 2: (-18, 10), 3: (-62, 2)}
labels = {1: "Kankava", 2: "Cork", 3: "Gueye"}
for _, row in top3.iterrows():
    r = int(row["rank"])
    ax.annotate(
        f"#{r} {labels[r]}",
        (row["style_similarity"], row["quality_similarity"]),
        textcoords="offset points",
        xytext=offsets[r],
        fontsize=12,
        fontweight="bold",
        color=INK,
        zorder=6,
    )
ax.text(0.03, 0.97, f"{int(above.sum())} closer on quality", transform=ax.transAxes,
        ha="left", va="top", color=BAD, fontsize=12, fontweight="bold")
ax.text(0.97, 0.03, f"{int(below.sum())} closer on style", transform=ax.transAxes,
        ha="right", va="bottom", color=GOOD, fontsize=12, fontweight="bold")
ax.text(93, 89.2, "balanced", rotation=45, rotation_mode="anchor", ha="center",
        color=MUTED, fontsize=11, style="italic")
ax.set_xlim(lim0, lim1)
ax.set_ylim(lim0, lim1)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Style similarity to Kanté", fontsize=13, color=INK)
ax.set_ylabel("Quality similarity to Kanté", fontsize=13, color=INK)
ax.tick_params(labelsize=11, colors=MUTED)
ax.grid(True, color="#e4e0da", linewidth=0.8, zorder=0)
for spine in ax.spines.values():
    spine.set_color(RULE)
_save(fig, "similarity-scatter.png")


def _clock(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value)
    if text.lower() in {"none", "nan", ""}:
        return None
    m, s = text.split(":")
    return float(m) * 60 + float(s)


def _parse_positions(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return ast.literal_eval(value)
    return []


def _minutes_played(positions, match_end):
    spans = []
    for pos in positions or []:
        start = _clock(pos.get("from"))
        end = _clock(pos.get("to")) or match_end
        if start is not None and end and end > start:
            spans.append((start, end))
    if not spans:
        return 0.0
    spans.sort()
    merged = [list(spans[0])]
    for s, e in spans[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return sum(e - s for s, e in merged) / 60.0


matches_m = pd.read_parquet(DATA_DIR / "matches.parquet")
lineups_m = pd.read_parquet(DATA_DIR / "lineups.parquet")
ev_m = pd.read_parquet(DATA_DIR / "events.parquet", columns=["match_id", "minute", "second"])
matches_m["match_date"] = pd.to_datetime(matches_m["match_date"])
end = (
    ev_m.assign(t=ev_m["minute"].astype(float) * 60 + ev_m["second"].astype(float))
    .groupby("match_id")["t"]
    .max()
)
matches_m["match_end"] = matches_m["match_id"].map(end).fillna(90 * 60)
matches_m["length"] = (matches_m["match_end"] / 60.0).clip(lower=90.0)

for key, idx in players:
    if key == "kante":
        continue
    row = chart_pool.loc[idx]
    teams = [t.strip() for t in str(row["teams"]).split(",")]
    team_matches = matches_m[matches_m["home_team"].isin(teams) | matches_m["away_team"].isin(teams)]
    team_matches = team_matches.sort_values("match_date")
    pl = lineups_m[lineups_m["player_id"] == row["player_id"]].set_index("match_id")["positions"]
    shares = []
    for m in team_matches.itertuples(index=False):
        mins = _minutes_played(_parse_positions(pl.get(m.match_id)), m.match_end) if m.match_id in pl.index else 0.0
        shares.append(min(mins / m.length, 1.0))
    n = len(shares)
    fig, ax = plt.subplots(figsize=(5.6, 1.25))
    x = np.arange(n)
    ax.bar(x, np.ones(n), width=0.82, color="#e4e0da", edgecolor="none", zorder=1)
    ax.bar(x, shares, width=0.82, color=GOOD, edgecolor="none", zorder=2)
    ticks = [0, n // 2, n - 1]
    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [team_matches.iloc[i]["match_date"].strftime("%b %Y") for i in ticks],
        fontsize=11,
        color=MUTED,
    )
    ax.set_xlim(-0.7, n - 0.3)
    ax.set_ylim(0, 1.02)
    ax.set_yticks([])
    ax.tick_params(axis="x", length=0, pad=4)
    for spine in ax.spines.values():
        spine.set_visible(False)
    _save(fig, f"minutes-{key}.png")

needed = [
    "similarity-scatter.png",
    "style-kante.png",
    "style-twin1.png",
    "style-twin2.png",
    "style-twin3.png",
    "quality-kante.png",
    "quality-twin1.png",
    "quality-twin2.png",
    "quality-twin3.png",
    "minutes-twin1.png",
    "minutes-twin2.png",
    "minutes-twin3.png",
]
missing = [name for name in needed if not (ASSETS_DIR / name).exists()]
print("assets ready:" if not missing else "missing:", missing or len(needed))


Wrote slides\kante-replacement\assets\style-kante.png
Wrote slides\kante-replacement\assets\quality-kante.png
Wrote slides\kante-replacement\assets\style-twin1.png
Wrote slides\kante-replacement\assets\quality-twin1.png
Wrote slides\kante-replacement\assets\style-twin2.png
Wrote slides\kante-replacement\assets\quality-twin2.png
Wrote slides\kante-replacement\assets\style-twin3.png
Wrote slides\kante-replacement\assets\quality-twin3.png
Wrote slides\kante-replacement\assets\similarity-scatter.png
Wrote slides\kante-replacement\assets\minutes-twin1.png
Wrote slides\kante-replacement\assets\minutes-twin2.png
Wrote slides\kante-replacement\assets\minutes-twin3.png
assets ready: 12
